# Lab 10-02 — Multi-hop retrieval loop (HotpotQA-style questions)

**Track 10 · Agentic RAG** — how retrieval becomes an iterative, budget-bounded process.

Single-shot retrieval fails when the answer requires combining facts from several documents. A multi-hop loop iteratively decomposes the question, retrieves, rephrases the open sub-question using what it learned, retrieves again, until it can answer. Think of it as tool-calling lab 01 but with retrieval as the only tool and an explicit hop budget.

```text
HotpotQA dev set (3 questions, 10 candidate paragraphs each)
  -> BGE embed the 10 paragraphs per question (local, cpu)
  -> hop 0: embed raw question -> cosine top-2 unseen paragraphs
  -> hop 1..N: LLM rewrites sub-query (json_object) -> embed -> cosine top-2 unseen
  -> termination: LLM says done or MAX_HOPS reached
  -> LLM generates final answer from all accumulated evidence
  -> verification gate (--verify)
```

Each HotpotQA question carries 10 candidate paragraphs (2 gold + 8 distractors). The loop runs up to `MAX_HOPS = 3` hops, each pulling `RETRIEVE_K = 2` paragraphs. The LLM decides when to stop (done) or what to search for next, turning retrieval into an agentic process.


## Setup

This notebook mirrors `curriculum/10-agentic-rag/02-multi-hop-loop.py` exactly: the same verified code, split into cells. Two prerequisites must hold before it will run:

- **Ollama serving `qwen2.5-coder:7b` at `localhost:11434`** — the local LLM every hop's sub-query rewrite and final answer generation goes to (`llms/ollama.py` talks to it through `langchain-ollama`). Fully local: no API key, no quota. If the server is not up, every LLM call fails and the loop never completes.
- **The data file on disk** — `Data/corpus/hotpotqa/hotpot_dev_distractor_v1.json`, already fetched by the repo's manifest-verified fetchers.

The imports cell walks up to the repo root and cd's into it, because a notebook has no `__file__` — so every `Data/...` path resolves exactly like the lab script. From the terminal the lab runs as:

```bash
python curriculum/10-agentic-rag/02-multi-hop-loop.py          # run + demo
python curriculum/10-agentic-rag/02-multi-hop-loop.py --verify # verification gate
```

The next cell installs the lab-specific dependencies (a no-op if you already ran `pip install -r requirements.txt`).


In [ ]:
# Lab-specific dependencies (already in requirements.txt -- the install
# is a no-op safety net for fresh environments):
#   langchain-ollama      -> the Ollama chat backend behind llms/ollama.py
#   langchain-huggingface -> BGE embedding via embeddings/bge.py
%pip install langchain-ollama langchain-huggingface


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the
# kernel's working directory -- this works whether the kernel launches
# from the repo root (like the lab script) or from the notebook's own
# folder (Jupyter's default) -- then cd into it so every repo-relative
# path behaves exactly like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

from embeddings.bge import BGEEmbedding  # noqa: E402
from llms.ollama import OllamaLLM  # noqa: E402


## 1. Configuration

Everything that keeps this lab fast but still meaningful is a named constant. `N_QUESTIONS = 3` takes three HotpotQA dev questions, and because each question costs ~3-4 LLM calls (one per hop plus the final answer), this number *is* the runtime knob. `MAX_HOPS = 3` caps the retrieval loop so it cannot spin forever. `RETRIEVE_K = 2` pulls two paragraphs per hop, small enough to stay focused but enough to accumulate evidence. `BGE_DEVICE = "cpu"` because Ollama holds most of the GPU VRAM for the LLM.


In [ ]:
# --------------------------------------------------------------------------
# 1. Configuration
# --------------------------------------------------------------------------
HOTPOT_PATH = Path("Data/corpus/hotpotqa/hotpot_dev_distractor_v1.json")
N_QUESTIONS = 3  # each question costs ~3-4 LLM calls; keep the lab fast
MAX_HOPS = 3  # retrieval hops per question (hop 0 uses the raw question)
RETRIEVE_K = 2  # paragraphs pulled per hop
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DEVICE = "cpu"  # shared GPU: Ollama holds most of VRAM


## 2. Load — first N HotpotQA questions

Each HotpotQA question comes with 10 candidate paragraphs (2 gold + 8 distractors), a question string, a gold answer, and supporting-fact titles. `load_questions` reads the JSON dev set and returns the first `n` records. `question_passages` extracts the paragraph texts and titles from a record. `gold_paragraphs` returns the set of paragraph titles marked as required evidence. `answer_contains` is a normalized substring check: did the final answer include the gold answer?


In [ ]:
# --------------------------------------------------------------------------
# 2. Load — first N HotpotQA questions (each has 10 context paragraphs)
# --------------------------------------------------------------------------
def load_questions(path: Path, n: int) -> list[dict]:
    """Return the first ``n`` questions from the dev set."""
    with open(path) as f:
        records = json.load(f)
    return records[:n]


def question_passages(rec: dict) -> tuple[list[str], list[str]]:
    """Return (passage_texts, paragraph_titles) for the 10 context paragraphs."""
    texts: list[str] = []
    titles: list[str] = []
    for title, sentences in rec["context"]:
        titles.append(title)
        texts.append(" ".join(sentences))
    return texts, titles


def gold_paragraphs(rec: dict) -> set[str]:
    """The paragraph titles HotpotQA marks as required evidence."""
    return {title for title, _ in rec["supporting_facts"]}


def answer_contains(gold: str, answer: str) -> bool:
    """Normalized substring check: is the gold answer inside the answer?"""
    return gold.strip().lower() in answer.strip().lower()


## 3. Experiment — iterative retrieve-then-refine loop per question

This is the multi-hop retrieval loop, in three parts:

- **Cosine retrieval** -- `_top_unseen` embeds a query with BGE, ranks all 10 candidate paragraphs by cosine similarity, and returns the top-k indices not yet seen. No vector database needed; the 10 paragraphs per question are small enough to rank in memory.
- **Sub-query rewriting** -- `_next_search_need` asks the LLM (via `json_object`) whether the evidence so far is sufficient. If not, it returns a rewritten sub-query targeting the missing fact. If sufficient (done), it returns None to stop hopping.
- **Per-question loop** -- `run_one_question` ties it together: hop 0 retrieves with the raw question, each later hop rewrites and retrieves again, and when the loop ends the LLM generates the final answer from all accumulated evidence.

`run_experiment` runs this loop over `N_QUESTIONS` questions, times it, and returns the rows plus aggregate stats (total LLM calls, how many questions hit a gold paragraph).


In [ ]:
# --------------------------------------------------------------------------
# 3. Experiment — iterative retrieve-then-refine loop per question
# --------------------------------------------------------------------------
def _cosine(a: list[float], b: list[float]) -> float:
    norm_a = sum(x * x for x in a) ** 0.5
    norm_b = sum(x * x for x in b) ** 0.5
    if norm_a == 0.0 or norm_b == 0.0:
        return 0.0
    return sum(x * y for x, y in zip(a, b)) / (norm_a * norm_b)


def _top_unseen(query: str, vecs: list[list[float]], embedder,
                seen: set[int], k: int) -> list[int]:
    """Cosine top-k indices over ``vecs``, skipping indices already seen."""
    query_vec = embedder.embed_documents([query])[0]
    ranked = sorted(
        range(len(vecs)),
        key=lambda i: _cosine(query_vec, vecs[i]),
        reverse=True,
    )
    return [i for i in ranked if i not in seen][:k]


def _next_search_need(llm, question: str, evidence: list[dict]) -> str | None:
    """Ask the LLM for the next sub-query; None means 'enough evidence'."""
    evidence_block = "\n".join(
        f"- {item['title']}: {item['text'][:180]}..." for item in evidence
    ) or "- (none yet)"
    prompt = (
        "You are doing multi-hop retrieval for a question.\n"
        "Given the question and the evidence gathered so far, decide the "
        "next search need.\n"
        "Rules:\n"
        '- If the evidence already suffices to answer, respond '
        '{"done": true, "search_need": ""}.\n'
        '- Otherwise respond {"done": false, "search_need": "a short, '
        'specific query for the missing fact"}.\n'
        "- Output ONLY JSON.\n\n"
        f"Question: {question}\n\n"
        f"Evidence so far:\n{evidence_block}"
    )
    result = llm.json_object(prompt)
    if not isinstance(result, dict) or "error" in result:
        return None  # cannot refine -> stop hopping
    done = result.get("done", False)
    if done is True or str(done).strip().lower() in ("true", "1", "yes"):
        return None
    need = str(result.get("search_need", result.get("sub_query", ""))).strip()
    return need or None


def run_one_question(rec: dict, llm, embedder) -> dict:
    passages, titles = question_passages(rec)
    vecs = embedder.embed_documents(passages)  # embed the 10 paragraphs once
    gold = rec["answer"]
    gold_titles = gold_paragraphs(rec)

    evidence: list[dict] = []
    hops: list[tuple[str, list[str]]] = []
    seen: set[int] = set()
    llm_calls = 0
    sub_query = rec["question"]
    for hop in range(MAX_HOPS):
        hits = _top_unseen(sub_query, vecs, embedder, seen, RETRIEVE_K)
        if not hits:
            break
        seen.update(hits)
        evidence.extend(
            {"title": titles[i], "text": passages[i]} for i in hits
        )
        hops.append((sub_query, [titles[i] for i in hits]))
        if hop == MAX_HOPS - 1:
            break
        llm_calls += 1
        next_query = _next_search_need(llm, rec["question"], evidence)
        if next_query is None:
            break
        sub_query = next_query

    evidence_block = "\n\n".join(
        f"[{item['title']}] {item['text']}" for item in evidence
    )
    llm_calls += 1
    answer = llm.invoke(
        "Answer the multi-hop question using ONLY the evidence passages "
        "below. If the evidence is insufficient, say what is missing.\n\n"
        f"Evidence:\n{evidence_block}\n\n"
        f"Question: {rec['question']}\n\n"
        "Answer in one or two sentences."
    ).strip()

    return {
        "question": rec["question"],
        "hops": hops,
        "evidence_titles": [titles[i] for i in seen],
        "answer": answer,
        "gold": gold,
        "gold_titles": sorted(gold_titles),
        "answer_contains_gold": answer_contains(gold, answer),
        "llm_calls": llm_calls,
    }


def run_experiment() -> dict:
    questions = load_questions(HOTPOT_PATH, N_QUESTIONS)
    llm = OllamaLLM()
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME, device=BGE_DEVICE)

    t0 = time.perf_counter()
    rows = [run_one_question(rec, llm, embedder) for rec in questions]
    total_s = time.perf_counter() - t0

    return {
        "rows": rows,
        "total_s": total_s,
        "agg": {
            "questions": len(rows),
            "total_llm_calls": sum(row["llm_calls"] for row in rows),
            "hits_gold": sum(
                bool(set(row["evidence_titles"]) & set(row["gold_titles"]))
                for row in rows
            ),
        },
    }


## 4. Demo

The demo prints the artifact from each question: the hop trace (what query retrieved what paragraphs at each hop), the evidence titles, the final answer, and whether it contains the gold answer. The takeaway: multi-hop retrieval is iterative refinement, not a single query. Each hop reuses what the previous one learned, and the LLM decides when to stop.


In [ ]:
# --------------------------------------------------------------------------
# 4. Demo — print the artifact
# --------------------------------------------------------------------------
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 10-02 — Iterative multi-hop retrieval loop (HotpotQA)")
    print(f"{exp['agg']['questions']} questions in {exp['total_s']:.1f}s, "
          f"{exp['agg']['total_llm_calls']} LLM calls")
    print("=" * 66)

    for i, row in enumerate(exp["rows"], start=1):
        print(f"\nQ{i}: {row['question'][:90]}")
        for hop, (sub_query, hit_titles) in enumerate(row["hops"], start=1):
            print(f"    hop {hop}: query {sub_query!r} -> {hit_titles}")
        print(f"    evidence    : {row['evidence_titles'][:4]}")
        print(f"    final answer: {row['answer'][:120]}")
        print(f"    gold answer : {row['gold']}  "
              f"(gold paras: {', '.join(row['gold_titles'])[:70]})")
        print(f"    gold in answer: {row['answer_contains_gold']}")

    a = exp["agg"]
    print(f"\n[5] Aggregates over {a['questions']} questions")
    print(f"    total LLM calls        : {a['total_llm_calls']}")
    print(f"    questions whose evidence touched a gold paragraph: "
          f"{a['hits_gold']}/{a['questions']}")

    print(f"\n[6] Takeaway")
    print("    Multi-hop retrieval is a loop, not a single query: each hop")
    print("    re-embeds a model-refined sub-query and grows the evidence.")
    print("    The LLM decides when to stop (done) — turning retrieval into")
    print("    an agentic, budget-bounded process. Answer quality is not")
    print("    gated here; termination, evidence accumulation, and a non-")
    print("    empty answer are.")


## 5. Verification gate

The gate checks structural invariants, not answer quality. A local 7B model may retrieve the wrong paragraphs and still pass. The checks: exactly `N_QUESTIONS` questions processed, every question has a non-empty hop trace, every question terminates within `MAX_HOPS` hops, every question accumulated at least one unique retrieved paragraph, and every final answer is non-empty. The gate turns "the lab ran" into "the lab ran *correctly*".


In [ ]:
# --------------------------------------------------------------------------
# 5. Verification gate — run ``python <lab> --verify`` from the repo root
# --------------------------------------------------------------------------
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []
    rows = exp["rows"]

    checks.append((f"exactly {N_QUESTIONS} questions processed",
                   exp["agg"]["questions"] == N_QUESTIONS))
    checks.append(("every question has a non-empty hop trace",
                   all(row["hops"] for row in rows)))
    checks.append((f"every question terminates within {MAX_HOPS} hops",
                   all(len(row["hops"]) <= MAX_HOPS for row in rows)))
    checks.append(("every question accumulated >= 1 unique retrieved paragraph",
                   all(row["evidence_titles"] for row in rows)))
    checks.append(("every final answer is non-empty",
                   all(row["answer"].strip() for row in rows)))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


## Run the experiment

3 questions with up to 3 hops each means a bounded number of LLM calls. Expect a few minutes depending on your hardware. `exp` holds everything the demo and gate need.


In [ ]:
exp = run_experiment()


### Demo -- the artifact

The hop traces, evidence, and final answers: the iterative retrieval artifact.


In [ ]:
print_demo(exp)


### Verification gate

Expect every check to PASS, the same gate the CI-style `--verify` run enforces. If any line shows FAIL, check that Ollama is serving `qwen2.5-coder:7b` and that `Data/corpus/hotpotqa/hotpot_dev_distractor_v1.json` is intact.


In [ ]:
verify_gate(exp)
